<a href="https://colab.research.google.com/github/sakshik05/TAE1/blob/main/TAE1ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
from dataclasses import dataclass
from typing import List, Optional
from enum import Enum
import matplotlib.pyplot as plt
import numpy as np

class TaskState(Enum):
    READY = "ready"
    RUNNING = "running"
    TERMINATED = "terminated"

@dataclass
class Task:
    id: int
    name: str
    priority: int
    state: TaskState
    cpu_time_used: float = 0.0
    total_cpu_needed: float = 0.0
    context_switch_count: int = 0
    cpu_register: List[float] = None

    def __post_init__(self):
        self.cpu_register = [random.random() for _ in range(8)]
        # Random CPU time needed (e.g., 20ms to 100ms)
        self.total_cpu_needed = random.uniform(0.02, 0.1)

class CPU:
    def __init__(self):
        self.current_task: Optional[Task] = None
        self.total_time = 0.0
        self.context_switches = 0
        self.total_switch_overhead = 0.0

    def context_switch(self, old_task: Optional[Task], new_task: Task):
        overhead = 0.0002  # Fixed 0.2ms overhead per switch
        if old_task:
            old_task.context_switch_count += 1

        self.context_switches += 1
        self.total_switch_overhead += overhead
        self.current_task = new_task
        new_task.state = TaskState.RUNNING
        return overhead

class Scheduler:
    def __init__(self, quantum: float = 0.01):
        self.quantum = quantum
        self.ready_queue: List[Task] = []
        self.cpu = CPU()
        self.completed_tasks: List[Task] = []

    def add_task(self, task: Task):
        self.ready_queue.append(task)

    def schedule(self):
        while self.ready_queue or self.cpu.current_task:
            # If CPU is idle, pick a task
            if not self.cpu.current_task and self.ready_queue:
                next_task = self.ready_queue.pop(0)
                self.cpu.total_time += self.cpu.context_switch(None, next_task)

            if self.cpu.current_task:
                task = self.cpu.current_task
                remaining = task.total_cpu_needed - task.cpu_time_used
                run_duration = min(self.quantum, remaining)

                # Advance Virtual Clock (No sleeping!)
                self.cpu.total_time += run_duration
                task.cpu_time_used += run_duration

                if task.cpu_time_used >= task.total_cpu_needed:
                    task.state = TaskState.TERMINATED
                    self.completed_tasks.append(task)
                    self.cpu.current_task = None
                elif self.ready_queue:
                    # Preempt and switch to next in queue
                    old_task = task
                    old_task.state = TaskState.READY
                    self.ready_queue.append(old_task)
                    next_task = self.ready_queue.pop(0)
                    self.cpu.total_time += self.cpu.context_switch(old_task, next_task)

# Example Execution
scheduler = Scheduler(quantum=0.01)
for i in range(5):
    scheduler.add_task(Task(i, f"Task-{i}", random.randint(1,10), TaskState.READY))

scheduler.schedule()

print(f"Simulation Complete!")
print(f"Total Time: {scheduler.cpu.total_time:.4f}s")
print(f"Context Switches: {scheduler.cpu.context_switches}")
print(f"Efficiency: {(sum(t.total_cpu_needed for t in scheduler.completed_tasks)/scheduler.cpu.total_time)*100:.1f}%")


Simulation Complete!
Total Time: 0.4353s
Context Switches: 44
Efficiency: 98.0%


In [ ]:
import threading
import time

# Number of switches
SWITCH_COUNT = 100000

# Dummy task
def task(name):
    for i in range(SWITCH_COUNT):
        pass


def context_switch_simulator():
    t1 = threading.Thread(target=task, args=("Task 1",))
    t2 = threading.Thread(target=task, args=("Task 2",))

    start = time.perf_counter()

    t1.start()
    t2.start()

    t1.join()
    t2.join()

    end = time.perf_counter()

    total_time = end - start

    print("Total execution time:", total_time)
    print("Average switch overhead:", total_time / SWITCH_COUNT)


if __name__ == "__main__":
    context_switch_simulator()

Total execution time: 0.006431251999856613
Average switch overhead: 6.431251999856613e-08


In [ ]:
import threading
import time

# Number of iterations (workload)
ITERATIONS = 100_000_000

# Dummy task
def task():
    for i in range(ITERATIONS):
        x = i * i   # simple computation


# 🔹 Single-thread execution
start_single = time.perf_counter()

task()   # direct call (no context switching)

end_single = time.perf_counter()
single_time = end_single - start_single


# 🔹 Multi-thread execution (context switching)
t1 = threading.Thread(target=task)
t2 = threading.Thread(target=task)

start_multi = time.perf_counter()

t1.start()
t2.start()

t1.join()
t2.join()

end_multi = time.perf_counter()
multi_time = end_multi - start_multi


# 🔹 Calculate overhead
overhead = multi_time - single_time


# 🔹 Results
print("Single Thread Time:", single_time)
print("Multi Thread Time:", multi_time)
print("Context Switching Overhead:", overhead)

Single Thread Time: 5.9772263710001425
Multi Thread Time: 10.97313354300013
Context Switching Overhead: 4.995907171999988
